# Phase 1 — Data Audit, Validation & Preparation
## VoltRelay Energy Analytics | Data Analytics Hackathon

**Notebook:** `01_data_audit_and_preparation.ipynb`  
**Phase:** 1 of N  
**Status:** Data Foundation  

---

This notebook establishes a reliable, reproducible analytical foundation for the VoltRelay Energy hackathon project.  
It is **read-only with respect to raw data** — no source files in `data/raw/` are modified.  
All processed outputs are written to `data/interim/`, `data/processed/`, and `outputs/`.

---
## Section 1 · Project Introduction

**VoltRelay Energy** is a simulated EV battery-swapping network operating across **six Indian cities**:  
Bengaluru, Delhi NCR, Hyderabad, Mumbai, Pune, and Jaipur.

The dataset spans **January 2024 – June 2025** (~18 months) and contains approximately **3.9 million swap events** together with supporting operational data: station telemetry, rider profiles, battery inventory, support tickets, city-level context, and fleet-partner contracts.

### Dataset Ecosystem

| Dataset | Grain | Role |
|---|---|---|
| `swap_events` | One row per swap attempt | Core transactional log — revenue, failures, usage |
| `station_hourly_status` | One row per station per hour | Operational telemetry — inventory, power, faults |
| `riders` | One row per rider | Rider dimensions — cohort, segment, partner |
| `batteries` | One row per battery pack | Battery lifecycle — SoH, supplier, retirement |
| `stations` | One row per station | Station dimensions — city, generation, firmware |
| `support_tickets` | One row per ticket | Customer service — category, resolution, CSAT |
| `city_daily_context` | One row per city per day | External factors — weather, events, outages |
| `fleet_partners` | One row per partner | B2B contracts — pricing, segment, cities |

---
## Section 2 · Business Objectives

Three primary business challenges drive this analysis:

### Challenge 1 — Increasing Battery-Swap Failures
Swap failures directly reduce rider satisfaction and network revenue. Root causes may include station hardware generation, battery State-of-Health (SoH), firmware versions, telemetry gaps, or operational congestion.

### Challenge 2 — Declining Repeat Usage and Retention
New rider churn is costly. Retention patterns must be linked to sign-up channel, plan type, city, partner, and early-swap experience quality.

### Challenge 3 — Declining Profit Per Battery Swap
Revenue is under pressure from discounting, partner contract structures, and potentially increasing cost of failures. Pricing, tariff codes, and partner economics need quantification.

### Phase 1 Objective
Establish a clean, validated analytical foundation — not to answer business questions, but to ensure the data is trustworthy enough to answer them correctly in Phase 2+.

---
## Section 3 · Imports and Configuration

In [1]:
"""Phase 1 — Imports and path configuration.

Paths are configured using pathlib relative to this notebook's location,
ensuring compatibility with Google Colab (mount at /content/drive/...) and
local environments. Adjust ROOT_DIR if running from a different location.
"""
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import duckdb

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.width", 160)

# ---------------------------------------------------------------------------
# Path Configuration — adjust ROOT_DIR for Colab if needed
# ---------------------------------------------------------------------------
# Option A: Running locally (auto-detect from notebook location)
try:
    NOTEBOOK_DIR = Path(globals()["__vsc_ipynb_file__"]).parent
    ROOT_DIR = NOTEBOOK_DIR.parent
except KeyError:
    ROOT_DIR = Path(os.getcwd()).parent  # fallback: assume cwd is notebooks/

# Option B: Google Colab override — uncomment and set your Drive path:
# ROOT_DIR = Path("/content/drive/MyDrive/VOLT-SCOPE")

RAW_DIR       = ROOT_DIR / "data" / "raw"
INTERIM_DIR   = ROOT_DIR / "data" / "interim"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
TABLES_DIR    = ROOT_DIR / "outputs" / "tables"
REPORTS_DIR   = ROOT_DIR / "reports"

for d in [INTERIM_DIR, PROCESSED_DIR, TABLES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Python  : {sys.version.split()[0]}")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"DuckDB  : {duckdb.__version__}")
print(f"Root    : {ROOT_DIR}")
print(f"Raw data: {RAW_DIR}")

ModuleNotFoundError: No module named 'duckdb'

---
## Section 4 · Dataset Discovery

Identify all files present in `data/raw/` and check for the eight expected datasets.

In [ ]:
EXPECTED_DATASETS = [
    "swap_events.csv",
    "station_hourly_status.csv",
    "riders.csv",
    "batteries.csv",
    "support_tickets.csv",
    "stations.csv",
    "city_daily_context.csv",
    "fleet_partners.csv",
]

print("=" * 70)
print("DATASET DISCOVERY")
print("=" * 70)

discovery_rows = []
for name in EXPECTED_DATASETS:
    path = RAW_DIR / name
    present = path.is_file()
    size_mib = path.stat().st_size / (1024 * 1024) if present else None
    discovery_rows.append({
        "dataset": name,
        "present": present,
        "size_MiB": round(size_mib, 2) if size_mib else "N/A",
    })
    status = "✓" if present else "✗ MISSING"
    size_str = f"{size_mib:.1f} MiB" if size_mib else ""
    print(f"  {status}  {name:<40} {size_str}")

missing = [r["dataset"] for r in discovery_rows if not r["present"]]
print()
if missing:
    print(f"⚠ MISSING DATASETS ({len(missing)}): {missing}")
else:
    print("✓ All 8 expected datasets are present.")

# Also check for any unexpected files
all_files = list(RAW_DIR.glob("*"))
unexpected = [f.name for f in all_files if f.name not in EXPECTED_DATASETS]
if unexpected:
    print(f"\nℹ Additional files found: {unexpected}")

---
## Section 5 · Dataset Overview

Load pre-generated audit summaries from `outputs/tables/` (produced by `src/data/audit_data.py`).  
These summaries were computed by streaming the large CSVs in chunks — no full dataset is loaded here.

In [ ]:
# Load pre-computed audit summaries
def load_table(name: str) -> pd.DataFrame:
    """Load a CSV from outputs/tables/, return empty DataFrame if missing."""
    path = TABLES_DIR / name
    if not path.is_file():
        print(f"⚠ {name} not found — run src/data/audit_data.py first.")
        return pd.DataFrame()
    return pd.read_csv(path)

inventory_df     = load_table("dataset_inventory.csv")
missing_df       = load_table("missing_value_summary.csv")
key_df           = load_table("duplicate_key_summary.csv")
category_df      = load_table("categorical_value_summary.csv")
numeric_df       = load_table("numerical_summary.csv")
fk_df            = load_table("foreign_key_validation.csv")
quality_df       = load_table("data_quality_issue_summary.csv")

print("Audit summary tables loaded:")
for name, df in [("inventory", inventory_df), ("missing", missing_df),
                 ("keys", key_df), ("categories", category_df),
                 ("numerics", numeric_df), ("foreign_keys", fk_df),
                 ("quality", quality_df)]:
    print(f"  {name:<15}: {len(df):,} rows")

In [ ]:
# High-level inventory overview
if not inventory_df.empty:
    display_inv = inventory_df[["dataset", "rows", "columns", "size_mib"]].copy()
    display_inv["rows"] = display_inv["rows"].apply(lambda x: f"{x:,}")
    display_inv["size_mib"] = display_inv["size_mib"].apply(lambda x: f"{x:.1f} MiB")
    print("DATASET INVENTORY")
    print(display_inv.to_string(index=False))

---
## Section 6 · Individual Dataset Profiling

For each dataset: schema, row/column counts, date ranges, key validation, and representative samples.  
Small reference datasets are loaded fully; large operational datasets use sampled reads.

In [ ]:
def profile_small(name: str, key_cols: list[str], date_cols: list[str]) -> pd.DataFrame:
    """Fully load and profile a small reference dataset."""
    path = RAW_DIR / name
    df = pd.read_csv(path, low_memory=False, encoding="utf-8-sig")
    print(f"\n{'='*70}")
    print(f"PROFILE: {name}")
    print(f"{'='*70}")
    print(f"Shape        : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\nColumns      :")
    print(df.dtypes.to_string())
    print(f"\nMissing values:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_report = pd.DataFrame({"missing": missing, "pct": missing_pct})
    print(missing_report[missing_report["missing"] > 0].to_string())
    for col in date_cols:
        if col in df.columns:
            parsed = pd.to_datetime(df[col], errors="coerce")
            print(f"\nDate range [{col}]: {parsed.min()} -> {parsed.max()} | invalid: {parsed.isna().sum() - df[col].isna().sum()}")
    print(f"\nSample (first 3 rows):")
    print(df.head(3).to_string())
    # Key uniqueness
    if key_cols and all(c in df.columns for c in key_cols):
        dup_keys = df.duplicated(subset=key_cols).sum()
        null_keys = df[key_cols].isnull().any(axis=1).sum()
        print(f"\nKey [{'+'.join(key_cols)}]: null_key_rows={null_keys:,}, duplicate_key_rows={dup_keys:,}")
    return df

def profile_large_sample(name: str, key_cols: list[str], date_cols: list[str], n: int = 10_000):
    """Sample-profile a large dataset — does not load full file."""
    path = RAW_DIR / name
    df_sample = pd.read_csv(path, nrows=n, low_memory=False, encoding="utf-8-sig")
    print(f"\n{'='*70}")
    print(f"PROFILE (sample): {name}")
    print(f"{'='*70}")
    # Row count from audit inventory
    if not inventory_df.empty:
        row_count = inventory_df.loc[inventory_df["dataset"] == name, "rows"].values
        if len(row_count):
            print(f"Full row count   : {int(row_count[0]):,} (from audit)")
    print(f"Sample shape     : {df_sample.shape[0]:,} rows × {df_sample.shape[1]} columns")
    print(f"\nColumns:")
    print(df_sample.dtypes.to_string())
    print(f"\nSample missing values (first {n:,} rows):")
    missing = df_sample.isnull().sum()
    print(missing[missing > 0].to_string())
    for col in date_cols:
        if col in df_sample.columns:
            parsed = pd.to_datetime(df_sample[col], errors="coerce")
            print(f"\nDate range [{col}] in sample: {parsed.min()} -> {parsed.max()}")
    print(f"\nSample (first 3 rows):")
    print(df_sample.head(3).to_string())
    return df_sample

In [ ]:
# --- 6.1 fleet_partners (smallest — full load) ---
df_fleet = profile_small("fleet_partners.csv",
    key_cols=["partner_id"],
    date_cols=["contract_start_date", "amendment_date"])

In [ ]:
# --- 6.2 stations ---
df_stations = profile_small("stations.csv",
    key_cols=["station_id"],
    date_cols=["commissioned_date", "decommissioned_date",
               "firmware_updated_date", "competitor_within_1_5km_since"])

In [ ]:
# --- 6.3 city_daily_context ---
df_context = profile_small("city_daily_context.csv",
    key_cols=["city", "date"],
    date_cols=["date"])

In [ ]:
# --- 6.4 batteries ---
df_batteries = profile_small("batteries.csv",
    key_cols=["battery_id"],
    date_cols=["manufacture_date", "commission_date", "retired_date"])

In [ ]:
# --- 6.5 riders ---
df_riders = profile_small("riders.csv",
    key_cols=["rider_id"],
    date_cols=["signup_date"])

In [ ]:
# --- 6.6 support_tickets (medium — full load) ---
df_tickets = profile_small("support_tickets.csv",
    key_cols=["ticket_id"],
    date_cols=["created_ts"])

In [ ]:
# --- 6.7 station_hourly_status (large — sampled) ---
df_telemetry_sample = profile_large_sample("station_hourly_status.csv",
    key_cols=["station_id", "hour_start"],
    date_cols=["hour_start"])

In [ ]:
# --- 6.8 swap_events (largest — sampled) ---
df_swaps_sample = profile_large_sample("swap_events.csv",
    key_cols=["event_id"],
    date_cols=["event_ts"])

---
## Section 7 · Data-Quality Assessment

Consolidated findings from the audit summaries. Each issue is documented with count, severity, and recommended action.

In [ ]:
# 7.1 Candidate key / duplicate summary
print("KEY UNIQUENESS SUMMARY")
print("=" * 70)
if not key_df.empty:
    display(key_df)
print()
print("Interpretation:")
print("  null_key_rows           = rows where the candidate key contains NULLs")
print("  duplicate_key_rows      = rows with a repeated key (beyond first occurrence)")
print("  duplicate_full_row_fingerprints = fully identical rows (hash-based)")

In [ ]:
# 7.2 Missing values — columns with any missingness
print("MISSING VALUES (columns with missing > 0)")
print("=" * 70)
if not missing_df.empty:
    has_missing = missing_df[missing_df["missing_count"] > 0].copy()
    has_missing = has_missing.sort_values(["dataset", "missing_pct"], ascending=[True, False])
    display(has_missing.reset_index(drop=True))

In [ ]:
# 7.3 Data quality issue summary from audit
print("DATA QUALITY ISSUE COUNTS (from audit_data.py)")
print("=" * 70)
if not quality_df.empty:
    display(quality_df.sort_values("issue").reset_index(drop=True))

In [ ]:
# 7.4 Important categorical distributions (using fully-loaded small datasets)
print("CATEGORICAL DISTRIBUTIONS — KEY FIELDS")
print("=" * 70)

if not category_df.empty:
    focus_cols = [
        "event_type", "sync_mode", "tariff_code", "payment_mode",
        "station_firmware", "telemetry_status", "resolution_status",
        "firmware_version", "location_type", "charger_generation",
        "pack_type", "supplier", "plan_type", "signup_channel",
    ]
    for col in focus_cols:
        subset = category_df[category_df["column"] == col].copy()
        if not subset.empty:
            print(f"\n--- {col} ---")
            subset["pct"] = (subset["count"] / subset["count"].sum() * 100).round(2)
            print(subset[["dataset", "value", "count", "pct"]].to_string(index=False))

In [ ]:
# 7.5 Numerical distribution summary
print("NUMERICAL DISTRIBUTION SUMMARY")
print("=" * 70)
if not numeric_df.empty:
    cols_to_show = [
        "dataset", "column", "nonmissing_count", "minimum", "maximum", "mean",
        "p01_sampled", "p25_sampled", "median_sampled", "p75_sampled", "p99_sampled"
    ]
    available = [c for c in cols_to_show if c in numeric_df.columns]
    display(numeric_df[available].sort_values(["dataset", "column"]).reset_index(drop=True))

In [ ]:
# 7.6 Detailed quality investigation on loaded datasets
print("DETAILED QUALITY CHECKS — LOADED DATASETS")
print("=" * 70)

# Riders: home_city spelling variants
print("\n--- riders.home_city unique values ---")
city_counts = df_riders["home_city"].value_counts(dropna=False)
print(city_counts.to_string())

# Riders: partner_id missing
print(f"\nRiders with missing partner_id: {df_riders['partner_id'].isna().sum():,}")
print(f"Riders with partner_id:         {df_riders['partner_id'].notna().sum():,}")

# Batteries: active vs retired
print("\n--- batteries.retirement_reason value counts ---")
print(df_batteries["retirement_reason"].value_counts(dropna=False).to_string())

# Stations: firmware versions
print("\n--- stations.firmware_version value counts ---")
print(df_stations["firmware_version"].value_counts(dropna=False).to_string())

# Stations: charger_generation
print("\n--- stations.charger_generation value counts ---")
print(df_stations["charger_generation"].value_counts(dropna=False).to_string())

# Tickets: resolution_status
print("\n--- support_tickets.resolution_status value counts ---")
print(df_tickets["resolution_status"].value_counts(dropna=False).to_string())

# Tickets: CSAT score distribution
print("\n--- support_tickets.csat_score stats ---")
print(f"  Missing: {df_tickets['csat_score'].isna().sum():,} of {len(df_tickets):,} tickets")
print(df_tickets["csat_score"].describe().to_string())

# Tickets: CSAT missing by resolution_status
print("\n--- CSAT missing by resolution_status ---")
csat_by_res = df_tickets.groupby("resolution_status", dropna=False).agg(
    total=("ticket_id", "count"),
    csat_missing=("csat_score", lambda x: x.isna().sum())
)
csat_by_res["missing_pct"] = (csat_by_res["csat_missing"] / csat_by_res["total"] * 100).round(2)
print(csat_by_res.to_string())

---
## Section 8 · Relationship and Join Validation

Validate primary/foreign key relationships across all datasets.  
Uses DuckDB for efficient joins without loading full swap_events or station_hourly_status into memory.

In [ ]:
# 8.1 Foreign key validation from audit
print("FOREIGN KEY VALIDATION (from audit_data.py)")
print("=" * 70)
if not fk_df.empty:
    display(fk_df)

print("\nNote: unmatched_rows=0 indicates complete referential integrity on that relationship.")
print("Nullable FK columns (battery_id, station_id on tickets) exclude nulls from denominator.")

In [ ]:
# 8.2 Relationship structure documentation
print("ENTITY-RELATIONSHIP STRUCTURE")
print("=" * 70)
print("""
swap_events (fact, ~3.9M rows)
  ├─ rider_id      ->  riders.rider_id              (N:1, mandatory)
  ├─ station_id    ->  stations.station_id           (N:1, mandatory)
  ├─ battery_in_id ->  batteries.battery_id          (N:1, nullable)
  └─ battery_out_id->  batteries.battery_id          (N:1, nullable)

station_hourly_status (fact-like, dense time series)
  └─ station_id    ->  stations.station_id           (N:1, mandatory)

support_tickets (fact)
  ├─ rider_id      ->  riders.rider_id               (N:1, mandatory)
  ├─ station_id    ->  stations.station_id            (N:1, nullable)
  └─ battery_id    ->  batteries.battery_id           (N:1, nullable)

riders
  └─ partner_id    ->  fleet_partners.partner_id      (N:1, nullable — retail riders have no partner)

stations
  └─ city          ->  city_daily_context.city        (N:M at city-day grain)

Join cardinality warnings:
  • city_daily_context joins to stations/swaps via city + date — do NOT join on city alone
    or every swap event will multiply by all days (N × 549 days).
  • station_hourly_status joins to swap_events via station_id + hour — not station_id alone.
  • batteries appear in swap_events TWICE (in + out) — count battery appearances carefully.
""")

In [ ]:
# 8.3 DuckDB-based join validation — cardinality and row-count checks
con = duckdb.connect()

# Register small reference tables
con.register("riders", df_riders)
con.register("stations", df_stations)
con.register("batteries", df_batteries)
con.register("fleet_partners", df_fleet)
con.register("city_daily_context", df_context)
con.register("support_tickets", df_tickets)

print("JOIN CARDINALITY VALIDATION")
print("=" * 70)

# Check 1: riders -> fleet_partners (N:1)
res = con.execute("""
    SELECT
        COUNT(*) as total_riders,
        COUNT(fp.partner_id) as matched_to_partner,
        COUNT(*) - COUNT(fp.partner_id) as unmatched,
        SUM(CASE WHEN r.partner_id IS NULL THEN 1 ELSE 0 END) as null_partner_id
    FROM riders r
    LEFT JOIN fleet_partners fp ON r.partner_id = fp.partner_id
""").fetchdf()
print("\nriders -> fleet_partners:")
display(res)

# Check 2: support_tickets -> riders (N:1)
res2 = con.execute("""
    SELECT
        COUNT(*) as total_tickets,
        SUM(CASE WHEN r.rider_id IS NOT NULL THEN 1 ELSE 0 END) as matched_riders,
        SUM(CASE WHEN r.rider_id IS NULL THEN 1 ELSE 0 END) as unmatched_riders
    FROM support_tickets t
    LEFT JOIN riders r ON t.rider_id = r.rider_id
""").fetchdf()
print("\nsupport_tickets -> riders:")
display(res2)

# Check 3: support_tickets -> stations (nullable FK)
res3 = con.execute("""
    SELECT
        COUNT(*) as total_tickets,
        SUM(CASE WHEN t.station_id IS NULL THEN 1 ELSE 0 END) as null_station_id,
        SUM(CASE WHEN t.station_id IS NOT NULL AND s.station_id IS NULL THEN 1 ELSE 0 END) as unmatched_station,
        SUM(CASE WHEN s.station_id IS NOT NULL THEN 1 ELSE 0 END) as matched_station
    FROM support_tickets t
    LEFT JOIN stations s ON t.station_id = s.station_id
""").fetchdf()
print("\nsupport_tickets -> stations (nullable):")
display(res3)

# Check 4: stations -> city_daily_context (city dimension)
res4 = con.execute("""
    SELECT
        s.city,
        COUNT(*) as station_count,
        COUNT(DISTINCT cdc.city) as has_context
    FROM stations s
    LEFT JOIN (SELECT DISTINCT city FROM city_daily_context) cdc ON s.city = cdc.city
    GROUP BY s.city ORDER BY s.city
""").fetchdf()
print("\nstations -> city_daily_context (city match):")
display(res4)

# Check 5: Dangerous join — city alone (shows fan-out risk)
res5 = con.execute("""
    SELECT
        COUNT(DISTINCT cdc.date) as context_days,
        COUNT(DISTINCT s.station_id) as station_count,
        COUNT(DISTINCT cdc.city) as context_cities,
        COUNT(DISTINCT s.city) as station_cities
    FROM stations s
    JOIN city_daily_context cdc ON s.city = cdc.city
""").fetchdf()
print("\nFan-out check: stations JOIN city_daily_context ON city alone:")
print("(Each station × every context day for its city — always add date filter!)")
display(res5)

con.close()

In [ ]:
# 8.4 Offline near-duplicate events from quality summary
if not quality_df.empty:
    q = quality_df.set_index("issue")["affected_records"]
    offline_pairs   = q.get("offline_near_duplicate_pairs_120s", 0)
    offline_events  = q.get("offline_near_duplicate_events_120s", 0)
    offline_total   = q.get("offline_batch_rows", 0)
    print("OFFLINE SYNC NEAR-DUPLICATE SCREENING")
    print("=" * 70)
    print(f"Total offline_batch events         : {offline_total:,}")
    print(f"Near-duplicate candidate pairs (±120s): {offline_pairs:,}")
    print(f"Distinct offline events in those pairs : {offline_events:,}")
    print()
    if offline_pairs > 0:
        print("⚠ FINDING: Near-duplicate offline candidates exist.")
        print("  These share the same rider, station, event_type, battery IDs,")
        print("  and timestamps within ±120 seconds, with at least one offline_batch sync.")
        print("  DISPOSITION: Retained as candidates. Deduplication requires business-rule")
        print("  confirmation. These are NOT removed in Phase 1.")
    else:
        print("✓ No near-duplicate offline candidates detected.")

---
## Section 9 · Investigation of 24 Unmatched Events

**Background:** A previous investigation identified 24 swap events that could not be matched to station-hour telemetry. All 24 share `station_firmware = v3.2.0`, each event's date falls exactly one day before its station's `commissioned_date`, and no telemetry row exists for the event date — but a telemetry row does exist at event_ts + 5h30m (IST offset).

**This section reproduces that finding from the raw data without altering any timestamps.**

In [ ]:
# 9.1 Build stations lookup (commissioning dates) — already loaded
stations_lookup = df_stations[["station_id", "commissioned_date", "city", "firmware_version"]].copy()
stations_lookup["commissioned_date"] = pd.to_datetime(stations_lookup["commissioned_date"], errors="coerce")

print(f"Stations loaded: {len(stations_lookup):,}")
print(f"Stations with valid commissioning date: {stations_lookup['commissioned_date'].notna().sum():,}")
print(f"\nFirmware version distribution:")
print(stations_lookup["firmware_version"].value_counts().to_string())

In [ ]:
# 9.2 Read swap_events in chunks — collect only v3.2.0 firmware events
print("Scanning swap_events.csv for station_firmware=v3.2.0 events...")

v320_records = []
CHUNK_SIZE = 100_000

for chunk in pd.read_csv(
    RAW_DIR / "swap_events.csv",
    chunksize=CHUNK_SIZE,
    low_memory=False,
    encoding="utf-8-sig",
    usecols=["event_id", "station_id", "event_ts", "station_firmware", "sync_mode", "event_type"]
):
    mask = chunk["station_firmware"].astype("string").eq("v3.2.0")
    if mask.any():
        v320_records.append(chunk[mask].copy())

if v320_records:
    df_v320 = pd.concat(v320_records, ignore_index=True)
    df_v320["event_ts"] = pd.to_datetime(df_v320["event_ts"], errors="coerce")
    df_v320["event_date"] = df_v320["event_ts"].dt.normalize()
    print(f"Total v3.2.0 firmware events: {len(df_v320):,}")
    print(f"Date range: {df_v320['event_ts'].min()} -> {df_v320['event_ts'].max()}")
else:
    df_v320 = pd.DataFrame()
    print("No v3.2.0 firmware events found.")

In [ ]:
# 9.3 Identify events where event_date = commissioned_date - 1 day
if not df_v320.empty:
    df_v320_ext = df_v320.merge(
        stations_lookup[["station_id", "commissioned_date"]],
        on="station_id", how="left"
    )

    # Events where the event date is exactly one day before commissioning
    ONE_DAY = pd.Timedelta(days=1)
    pre_commission_mask = (
        df_v320_ext["event_date"].notna() &
        df_v320_ext["commissioned_date"].notna() &
        (df_v320_ext["commissioned_date"] - df_v320_ext["event_date"] == ONE_DAY)
    )

    df_pre_commission = df_v320_ext[pre_commission_mask].copy()
    print(f"v3.2.0 events where event_date = commissioned_date - 1 day: {len(df_pre_commission):,}")

    if not df_pre_commission.empty:
        print(f"\nStation-days affected:")
        agg = df_pre_commission.groupby(["station_id", "event_date", "commissioned_date"]).size().reset_index(name="event_count")
        display(agg)
        print(f"\nTotal events: {df_pre_commission['event_id'].nunique():,}")
        print(f"Distinct stations: {df_pre_commission['station_id'].nunique():,}")
        print(f"Distinct event-dates: {df_pre_commission['event_date'].nunique():,}")

In [ ]:
# 9.4 Verify telemetry absence on event_date and presence at +5h30m
if not df_v320.empty and not df_pre_commission.empty:
    # Collect telemetry for the affected stations
    affected_stations = df_pre_commission["station_id"].unique().tolist()
    affected_dates    = df_pre_commission["event_date"].dt.date.unique().tolist()
    commission_dates  = df_pre_commission["commissioned_date"].dt.date.unique().tolist()

    print(f"Collecting telemetry for {len(affected_stations)} station(s) around affected dates...")

    tele_records = []
    for chunk in pd.read_csv(
        RAW_DIR / "station_hourly_status.csv",
        chunksize=100_000,
        low_memory=False,
        encoding="utf-8-sig",
        usecols=["station_id", "hour_start", "telemetry_status"]
    ):
        mask = chunk["station_id"].isin(affected_stations)
        if mask.any():
            tele_records.append(chunk[mask].copy())

    if tele_records:
        df_tele_affected = pd.concat(tele_records, ignore_index=True)
        df_tele_affected["hour_start"] = pd.to_datetime(df_tele_affected["hour_start"], errors="coerce")
        df_tele_affected["tele_date"] = df_tele_affected["hour_start"].dt.normalize()
        print(f"\nTelemetry rows for affected stations: {len(df_tele_affected):,}")
        print("\nTelemetry dates available per station:")
        tele_dates_per_station = df_tele_affected.groupby("station_id")["tele_date"].agg(["min", "max", "nunique"])
        display(tele_dates_per_station)
    else:
        df_tele_affected = pd.DataFrame()
        print("No telemetry found for affected stations.")

In [ ]:
# 9.5 Detailed telemetry vs event-date check
if not df_v320.empty and not df_pre_commission.empty and not df_tele_affected.empty:
    IST_OFFSET = pd.Timedelta(hours=5, minutes=30)

    print("TELEMETRY COVERAGE CHECK — AFFECTED STATION-DAYS")
    print("=" * 70)

    for _, row in agg.iterrows():
        sid = row["station_id"]
        evt_date = pd.Timestamp(row["event_date"]).normalize()
        comm_date = pd.Timestamp(row["commissioned_date"]).normalize()

        # Telemetry on event date
        tele_on_event_date = df_tele_affected[
            (df_tele_affected["station_id"] == sid) &
            (df_tele_affected["tele_date"] == evt_date)
        ]

        # Telemetry on commissioning date (the diagnostic +5h30 date)
        tele_on_comm_date = df_tele_affected[
            (df_tele_affected["station_id"] == sid) &
            (df_tele_affected["tele_date"] == comm_date)
        ]

        print(f"\nStation: {sid}")
        print(f"  Event date       : {evt_date.date()} | Telemetry rows on event date: {len(tele_on_event_date):,}")
        print(f"  Commissioned date: {comm_date.date()} | Telemetry rows on comm date : {len(tele_on_comm_date):,}")

        # Check if any event timestamp + 5h30 falls into commissioning-date telemetry
        events_this_station = df_pre_commission[df_pre_commission["station_id"] == sid]
        if not events_this_station.empty and not tele_on_comm_date.empty:
            tele_hours = set(df_tele_affected[
                (df_tele_affected["station_id"] == sid) &
                (df_tele_affected["tele_date"] == comm_date)
            ]["hour_start"].dt.floor("h").unique())

            shifted_match = 0
            for ts in events_this_station["event_ts"].dropna():
                shifted = (ts + IST_OFFSET).floor("h")
                if shifted in tele_hours:
                    shifted_match += 1
            print(f"  Events with event_ts+5h30 matching a telemetry hour: {shifted_match} of {len(events_this_station)}")

In [ ]:
# 9.6 Document and preserve the 24 events as traceable exceptions
if not df_v320.empty and not df_pre_commission.empty:
    # Save these 24 events to interim for traceability
    exception_path = INTERIM_DIR / "chronology_exceptions_24_events.parquet"
    df_pre_commission.to_parquet(exception_path, index=False)

    print("CHRONOLOGY EXCEPTION SUMMARY")
    print("=" * 70)
    print(f"Events preserved as exceptions : {len(df_pre_commission):,}")
    print(f"All belong to firmware version  : v3.2.0")
    print(f"Distinct stations affected      : {df_pre_commission['station_id'].nunique():,}")
    print(f"Distinct station-days           : {df_pre_commission.groupby(['station_id','event_date']).ngroups}")
    print()
    print("EVIDENCE:")
    print("  1. Each event's calendar date = its station's commissioned_date minus 1 day")
    print("  2. No station-hour telemetry exists for the event date")
    print("  3. Telemetry exists on the commissioned_date (the 'next day')")
    print("  4. event_ts + 5h30 (IST offset) maps to an hour that exists in telemetry")
    print()
    print("POSSIBLE EXPLANATIONS (not confirmed):")
    print("  A. Timestamps stored in UTC in v3.2.0 firmware, not IST — the event actually")
    print("     occurred on the commissioning date in local time")
    print("  B. Station commissioned and operated same day but commissioning date was")
    print("     recorded as following calendar day")
    print("  C. Clock synchronization issue in v3.2.0 firmware")
    print()
    print("DISPOSITION:")
    print("  - Events are RETAINED in full analysis (not deleted)")
    print("  - Timestamps are NOT corrected without explicit business-rule confirmation")
    print("  - Commissioning dates are NOT shifted")
    print(f"  - Exception file saved: {exception_path}")
    print()
    print("Full exception records:")
    display(df_pre_commission.sort_values(["station_id", "event_ts"]).reset_index(drop=True))

---
## Section 10 · Data Cleaning

Only necessary, defensible transformations are applied. Raw files are never modified.  
Cleaned outputs are written to `data/interim/` (cleaned reference tables) and `data/processed/` (analytical-ready datasets).

In [ ]:
# 10.1 Cleaning log — track every transformation
cleaning_log = []

def log_clean(dataset, transformation, rows_affected, rationale):
    cleaning_log.append({
        "dataset": dataset,
        "transformation": transformation,
        "rows_affected": rows_affected,
        "rationale": rationale,
    })
    print(f"  [{dataset}] {transformation} — {rows_affected:,} rows — {rationale}")

print("CLEANING LOG (all transformations recorded)")
print("=" * 70)

In [ ]:
# 10.2 fleet_partners — clean and save
fleet_clean = df_fleet.copy()

# Parse dates
for col in ["contract_start_date", "amendment_date"]:
    fleet_clean[col] = pd.to_datetime(fleet_clean[col], errors="coerce")
log_clean("fleet_partners", "parse date columns", len(fleet_clean), "Standardize to datetime")

# Strip whitespace from string cols
str_cols = fleet_clean.select_dtypes(include="object").columns
for col in str_cols:
    fleet_clean[col] = fleet_clean[col].str.strip()
log_clean("fleet_partners", "strip string whitespace", len(fleet_clean), "Defensive normalization")

print(f"\nfleet_partners cleaned: {len(fleet_clean)} rows")
fleet_clean.to_parquet(INTERIM_DIR / "fleet_partners_clean.parquet", index=False)
print(f"Saved -> data/interim/fleet_partners_clean.parquet")

In [ ]:
# 10.3 stations — clean and save
stations_clean = df_stations.copy()

# Parse date columns
date_cols = ["commissioned_date", "decommissioned_date", "firmware_updated_date", "competitor_within_1_5km_since"]
for col in date_cols:
    stations_clean[col] = pd.to_datetime(stations_clean[col], errors="coerce")
log_clean("stations", "parse date columns", len(stations_clean), "Standardize to datetime")

# Strip whitespace from string cols
str_cols = stations_clean.select_dtypes(include="object").columns
for col in str_cols:
    stations_clean[col] = stations_clean[col].str.strip()
log_clean("stations", "strip string whitespace", len(stations_clean), "Defensive normalization")

# Add derived: is_decommissioned flag
stations_clean["is_decommissioned"] = stations_clean["decommissioned_date"].notna()
log_clean("stations", "add is_decommissioned flag", len(stations_clean), "Derived field for active/inactive filtering")

print(f"\nstations cleaned: {len(stations_clean)} rows")
print(f"  Active stations: {(~stations_clean['is_decommissioned']).sum():,}")
print(f"  Decommissioned : {stations_clean['is_decommissioned'].sum():,}")
stations_clean.to_parquet(INTERIM_DIR / "stations_clean.parquet", index=False)
print(f"Saved -> data/interim/stations_clean.parquet")

In [ ]:
# 10.4 city_daily_context — clean and save
context_clean = df_context.copy()

# Parse date
context_clean["date"] = pd.to_datetime(context_clean["date"], errors="coerce")
log_clean("city_daily_context", "parse date column", len(context_clean), "Standardize to datetime")

# Strip whitespace from city
context_clean["city"] = context_clean["city"].str.strip()
log_clean("city_daily_context", "strip city whitespace", len(context_clean), "Defensive normalization")

# Check for and report any duplicate city+date keys
dup_ctx = context_clean.duplicated(subset=["city", "date"]).sum()
if dup_ctx > 0:
    print(f"⚠ {dup_ctx} duplicate city+date keys — keeping first occurrence")
    context_clean = context_clean.drop_duplicates(subset=["city", "date"], keep="first")
    log_clean("city_daily_context", "remove duplicate city+date rows", dup_ctx, "Keep first occurrence; duplicates are identical rows")
else:
    print("✓ No duplicate city+date keys")

print(f"\ncity_daily_context cleaned: {len(context_clean)} rows")
context_clean.to_parquet(INTERIM_DIR / "city_daily_context_clean.parquet", index=False)
print(f"Saved -> data/interim/city_daily_context_clean.parquet")

In [ ]:
# 10.5 batteries — clean and save
batteries_clean = df_batteries.copy()

# Parse date columns
for col in ["manufacture_date", "commission_date", "retired_date"]:
    batteries_clean[col] = pd.to_datetime(batteries_clean[col], errors="coerce")
log_clean("batteries", "parse date columns", len(batteries_clean), "Standardize to datetime")

# Strip string whitespace
str_cols = batteries_clean.select_dtypes(include="object").columns
for col in str_cols:
    batteries_clean[col] = batteries_clean[col].str.strip()
log_clean("batteries", "strip string whitespace", len(batteries_clean), "Defensive normalization")

# Add derived: is_retired flag
batteries_clean["is_retired"] = batteries_clean["retired_date"].notna()
log_clean("batteries", "add is_retired flag", len(batteries_clean), "Convenience field for active/retired filtering")

print(f"\nbatteries cleaned: {len(batteries_clean)} rows")
print(f"  Active batteries : {(~batteries_clean['is_retired']).sum():,}")
print(f"  Retired batteries: {batteries_clean['is_retired'].sum():,}")
print(f"\ncurrent_soh_pct range: {batteries_clean['current_soh_pct'].min():.2f} to {batteries_clean['current_soh_pct'].max():.2f}")
batteries_clean.to_parquet(INTERIM_DIR / "batteries_clean.parquet", index=False)
print(f"Saved -> data/interim/batteries_clean.parquet")

In [ ]:
# 10.6 riders — clean and save
riders_clean = df_riders.copy()

# Parse date
riders_clean["signup_date"] = pd.to_datetime(riders_clean["signup_date"], errors="coerce")
log_clean("riders", "parse signup_date", len(riders_clean), "Standardize to datetime")

# Strip string whitespace
str_cols = riders_clean.select_dtypes(include="object").columns
for col in str_cols:
    riders_clean[col] = riders_clean[col].str.strip()
log_clean("riders", "strip string whitespace", len(riders_clean), "Defensive normalization")

# Document city spelling variants without changing them
import re
def normalize_city(s):
    return re.sub(r"[^a-z0-9]", "", str(s).casefold())

city_norm = riders_clean["home_city"].dropna().apply(normalize_city)
city_groups = {}
for orig, norm in zip(riders_clean["home_city"].dropna(), city_norm):
    city_groups.setdefault(norm, set()).add(orig)
variant_groups = {k: sorted(v) for k, v in city_groups.items() if len(v) > 1}
if variant_groups:
    print("⚠ City spelling variants detected (source values preserved):")
    for norm_key, variants in sorted(variant_groups.items()):
        print(f"  {norm_key}: {variants}")
else:
    print("✓ No city spelling variants detected.")

# Add signup cohort (year-month)
riders_clean["signup_cohort"] = riders_clean["signup_date"].dt.to_period("M").astype("string")
log_clean("riders", "add signup_cohort (YYYY-MM)", len(riders_clean), "Needed for retention cohort analysis in Phase 2")

print(f"\nriders cleaned: {len(riders_clean):,} rows")
riders_clean.to_parquet(INTERIM_DIR / "riders_clean.parquet", index=False)
print(f"Saved -> data/interim/riders_clean.parquet")

In [ ]:
# 10.7 support_tickets — clean and save
tickets_clean = df_tickets.copy()

# Parse timestamp
tickets_clean["created_ts"] = pd.to_datetime(tickets_clean["created_ts"], errors="coerce")
log_clean("support_tickets", "parse created_ts", len(tickets_clean), "Standardize to datetime")

# Strip string whitespace
str_cols = tickets_clean.select_dtypes(include="object").columns
for col in str_cols:
    if col != "rider_comment":  # preserve free-text as-is
        tickets_clean[col] = tickets_clean[col].str.strip()
log_clean("support_tickets", "strip whitespace (except rider_comment)", len(tickets_clean), "Preserve free-text; normalize categories")

# Document invalid timestamps
invalid_ts = tickets_clean["created_ts"].isna().sum()
if invalid_ts > 0:
    print(f"⚠ {invalid_ts:,} tickets have unparseable created_ts — retained, flagged")
    log_clean("support_tickets", "flag invalid created_ts", invalid_ts, "Retain; mark for exclusion in time-series analysis")

# Add date column for joining to city_daily_context
tickets_clean["ticket_date"] = tickets_clean["created_ts"].dt.normalize()
log_clean("support_tickets", "add ticket_date", len(tickets_clean), "Date-level field for joining to context")

print(f"\nsupport_tickets cleaned: {len(tickets_clean):,} rows")
tickets_clean.to_parquet(INTERIM_DIR / "support_tickets_clean.parquet", index=False)
print(f"Saved -> data/interim/support_tickets_clean.parquet")

In [ ]:
# 10.8 swap_events — clean via chunked processing, write to Parquet
# Large file: stream, clean, and write to processed/ as Parquet
# DuckDB will be used for analysis (avoids full in-memory load)

print("Cleaning swap_events.csv (chunked streaming)...")
CHUNK = 100_000
swaps_out = PROCESSED_DIR / "swap_events_clean.parquet"

total_in = 0
total_out = 0
invalid_ts_count = 0
soh_over_100 = 0
soc_over_100 = 0

first_chunk = True
writer = None

try:
    import pyarrow as pa
    import pyarrow.parquet as pq

    for chunk in pd.read_csv(
        RAW_DIR / "swap_events.csv",
        chunksize=CHUNK,
        low_memory=False,
        encoding="utf-8-sig"
    ):
        total_in += len(chunk)

        # Standardize timestamps
        chunk["event_ts"] = pd.to_datetime(chunk["event_ts"], errors="coerce")
        invalid_ts_count += chunk["event_ts"].isna().sum()

        # Add derived date fields
        chunk["event_date"]  = chunk["event_ts"].dt.normalize()
        chunk["event_hour"]  = chunk["event_ts"].dt.hour
        chunk["event_month"] = chunk["event_ts"].dt.to_period("M").astype("string")

        # Count anomalies (do NOT remove)
        for col in ["soh_in_pct", "soh_out_pct"]:
            vals = pd.to_numeric(chunk[col], errors="coerce")
            soh_over_100 += int((vals > 100).sum())
        for col in ["soc_in_pct", "soc_out_pct"]:
            vals = pd.to_numeric(chunk[col], errors="coerce")
            soc_over_100 += int((vals > 100).sum())

        # Strip string whitespace from key categorical columns
        for col in ["event_type", "tariff_code", "payment_mode", "station_firmware", "sync_mode"]:
            if col in chunk.columns:
                chunk[col] = chunk[col].astype("string").str.strip()

        total_out += len(chunk)

        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(swaps_out, table.schema, compression="snappy")
        writer.write_table(table)

    if writer:
        writer.close()

    log_clean("swap_events", "parse event_ts to datetime", invalid_ts_count, f"{invalid_ts_count:,} unparseable timestamps retained with NaT")
    log_clean("swap_events", "add event_date, event_hour, event_month", total_out, "Derived time fields for aggregation")
    log_clean("swap_events", "strip whitespace from categorical columns", total_out, "Defensive normalization")
    log_clean("swap_events", f"SOH>100: {soh_over_100:,} | SOC>100: {soc_over_100:,}", soh_over_100 + soc_over_100, "Anomalies documented, NOT removed — await BMS validation")

    print(f"\nswap_events: {total_in:,} in -> {total_out:,} out (no rows removed)")
    print(f"Invalid timestamps: {invalid_ts_count:,}")
    print(f"SOH>100 observations: {soh_over_100:,}")
    print(f"SOC>100 observations: {soc_over_100:,}")
    print(f"Saved -> data/processed/swap_events_clean.parquet")

except ImportError:
    print("⚠ pyarrow not available. Saving swap_events in CSV chunks to processed/ instead.")
    # Fallback: save as CSV (not ideal for large data)
    for i, chunk in enumerate(pd.read_csv(
        RAW_DIR / "swap_events.csv",
        chunksize=CHUNK, low_memory=False, encoding="utf-8-sig"
    )):
        chunk["event_ts"] = pd.to_datetime(chunk["event_ts"], errors="coerce")
        chunk["event_date"] = chunk["event_ts"].dt.normalize()
        mode = "w" if i == 0 else "a"
        chunk.to_csv(PROCESSED_DIR / "swap_events_clean.csv", mode=mode, header=(i == 0), index=False)

In [ ]:
# 10.9 station_hourly_status — clean via chunked processing
print("Cleaning station_hourly_status.csv (chunked streaming)...")

tele_out = PROCESSED_DIR / "station_hourly_status_clean.parquet"
tele_total_in = 0
tele_invalid_ts = 0
tele_writer = None

try:
    import pyarrow as pa
    import pyarrow.parquet as pq

    for chunk in pd.read_csv(
        RAW_DIR / "station_hourly_status.csv",
        chunksize=CHUNK,
        low_memory=False,
        encoding="utf-8-sig"
    ):
        tele_total_in += len(chunk)
        chunk["hour_start"] = pd.to_datetime(chunk["hour_start"], errors="coerce")
        tele_invalid_ts += chunk["hour_start"].isna().sum()
        chunk["station_id"] = chunk["station_id"].astype("string").str.strip()
        chunk["telemetry_date"] = chunk["hour_start"].dt.normalize()

        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if tele_writer is None:
            tele_writer = pq.ParquetWriter(tele_out, table.schema, compression="snappy")
        tele_writer.write_table(table)

    if tele_writer:
        tele_writer.close()

    log_clean("station_hourly_status", "parse hour_start to datetime", tele_invalid_ts, f"{tele_invalid_ts:,} invalid timestamps")
    log_clean("station_hourly_status", "add telemetry_date", tele_total_in, "Date field for daily aggregation")
    log_clean("station_hourly_status", "strip station_id whitespace", tele_total_in, "Defensive normalization")

    print(f"\nstation_hourly_status: {tele_total_in:,} rows processed")
    print(f"Invalid timestamps: {tele_invalid_ts:,}")
    print(f"Saved -> data/processed/station_hourly_status_clean.parquet")

except ImportError:
    print("⚠ pyarrow not available — station_hourly_status not saved to parquet.")
    print("  Install pyarrow for efficient storage. Analysis will read from raw CSV via DuckDB.")

In [ ]:
# 10.10 Print the complete cleaning log
print("COMPLETE CLEANING LOG")
print("=" * 70)
log_df = pd.DataFrame(cleaning_log)
display(log_df)
log_df.to_csv(TABLES_DIR / "cleaning_log.csv", index=False)
print(f"\nLog saved -> outputs/tables/cleaning_log.csv")

---
## Section 11 · Post-Cleaning Validation

Validate the cleaning outputs against expectations.  
Uses DuckDB to query the Parquet files directly — avoids full in-memory loads.

In [ ]:
# 11.1 Row count reconciliation
print("ROW COUNT RECONCILIATION")
print("=" * 70)

# Reference counts from audit inventory
audit_counts = {}
if not inventory_df.empty:
    for _, row in inventory_df.iterrows():
        audit_counts[row["dataset"]] = int(row["rows"])

# Count cleaned Parquet outputs
processed_files = {
    "swap_events.csv"           : PROCESSED_DIR / "swap_events_clean.parquet",
    "station_hourly_status.csv" : PROCESSED_DIR / "station_hourly_status_clean.parquet",
}
interim_files = {
    "fleet_partners.csv"        : INTERIM_DIR / "fleet_partners_clean.parquet",
    "stations.csv"              : INTERIM_DIR / "stations_clean.parquet",
    "city_daily_context.csv"    : INTERIM_DIR / "city_daily_context_clean.parquet",
    "batteries.csv"             : INTERIM_DIR / "batteries_clean.parquet",
    "riders.csv"                : INTERIM_DIR / "riders_clean.parquet",
    "support_tickets.csv"       : INTERIM_DIR / "support_tickets_clean.parquet",
}

con = duckdb.connect()

print(f"{'Dataset':<40} {'Raw rows':>12} {'Clean rows':>12} {'Match':>8}")
print("-" * 75)

for ds, path in {**processed_files, **interim_files}.items():
    if path.is_file():
        clean_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
        raw_count = audit_counts.get(ds, "N/A")
        match = "✓" if isinstance(raw_count, int) and clean_count == raw_count else "≠"
        # city_daily_context may have had duplicates removed
        if ds == "city_daily_context.csv":
            match = f"≠ (expected if dups removed)"
        print(f"{ds:<40} {str(raw_count):>12} {clean_count:>12,} {match:>8}")
    else:
        print(f"{ds:<40} {'N/A':>12} {'NOT FOUND':>12} {'⚠':>8}")

con.close()

In [ ]:
# 11.2 Schema validation on cleaned Parquet outputs
print("SCHEMA VALIDATION — CLEANED PARQUET FILES")
print("=" * 70)

con = duckdb.connect()

for label, path in [
    ("swap_events_clean",              PROCESSED_DIR / "swap_events_clean.parquet"),
    ("station_hourly_status_clean",    PROCESSED_DIR / "station_hourly_status_clean.parquet"),
    ("riders_clean",                   INTERIM_DIR / "riders_clean.parquet"),
    ("batteries_clean",                INTERIM_DIR / "batteries_clean.parquet"),
    ("stations_clean",                 INTERIM_DIR / "stations_clean.parquet"),
    ("support_tickets_clean",          INTERIM_DIR / "support_tickets_clean.parquet"),
    ("fleet_partners_clean",           INTERIM_DIR / "fleet_partners_clean.parquet"),
    ("city_daily_context_clean",       INTERIM_DIR / "city_daily_context_clean.parquet"),
]:
    if path.is_file():
        schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}') LIMIT 0").fetchdf()
        print(f"\n{label}:")
        print(schema[["column_name", "column_type"]].to_string(index=False))
    else:
        print(f"\n⚠ {label}: file not found")

con.close()

In [ ]:
# 11.3 Key uniqueness on cleaned reference tables
print("KEY UNIQUENESS — CLEANED REFERENCE TABLES")
print("=" * 70)

key_checks = [
    ("fleet_partners_clean",    INTERIM_DIR / "fleet_partners_clean.parquet",    "partner_id"),
    ("stations_clean",          INTERIM_DIR / "stations_clean.parquet",           "station_id"),
    ("batteries_clean",         INTERIM_DIR / "batteries_clean.parquet",          "battery_id"),
    ("riders_clean",            INTERIM_DIR / "riders_clean.parquet",             "rider_id"),
    ("support_tickets_clean",   INTERIM_DIR / "support_tickets_clean.parquet",    "ticket_id"),
    ("city_daily_context_clean",INTERIM_DIR / "city_daily_context_clean.parquet", "city,date"),
]

con = duckdb.connect()

for label, path, key in key_checks:
    if path.is_file():
        total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
        if "," in key:
            keys = key.replace(",", ", ")
            dupes = con.execute(f"SELECT COUNT(*) - COUNT(*) OVER () as dupes FROM (SELECT {keys}, COUNT(*) as cnt FROM read_parquet('{path}') GROUP BY {keys} HAVING cnt > 1)").fetchone()[0]
        else:
            dupes = con.execute(f"""
                SELECT COUNT(*) FROM (
                    SELECT {key}, COUNT(*) as cnt
                    FROM read_parquet('{path}')
                    GROUP BY {key} HAVING cnt > 1
                )
            """).fetchone()[0]
        status = "✓ UNIQUE" if dupes == 0 else f"⚠ {dupes} duplicate keys"
        print(f"{label:<35} rows={total:,} key=[{key}] -> {status}")
    else:
        print(f"{label:<35} ⚠ file not found")

con.close()

In [ ]:
# 11.4 Referential integrity check on cleaned data (DuckDB)
print("REFERENTIAL INTEGRITY — CLEANED DATA")
print("=" * 70)

con = duckdb.connect()

swaps_path   = str(PROCESSED_DIR / "swap_events_clean.parquet")
riders_path  = str(INTERIM_DIR / "riders_clean.parquet")
stations_path = str(INTERIM_DIR / "stations_clean.parquet")
batt_path    = str(INTERIM_DIR / "batteries_clean.parquet")

if all(Path(p).is_file() for p in [swaps_path, riders_path, stations_path, batt_path]):
    # swap_events -> riders
    r1 = con.execute(f"""
        SELECT COUNT(*) as unmatched_rider_ids
        FROM read_parquet('{swaps_path}') s
        WHERE s.rider_id NOT IN (SELECT rider_id FROM read_parquet('{riders_path}'))
    """).fetchone()[0]

    # swap_events -> stations
    r2 = con.execute(f"""
        SELECT COUNT(*) as unmatched_station_ids
        FROM read_parquet('{swaps_path}') s
        WHERE s.station_id NOT IN (SELECT station_id FROM read_parquet('{stations_path}'))
    """).fetchone()[0]

    # swap_events -> batteries (battery_in_id, non-null)
    r3 = con.execute(f"""
        SELECT COUNT(*) as unmatched_battery_in
        FROM read_parquet('{swaps_path}') s
        WHERE s.battery_in_id IS NOT NULL
        AND s.battery_in_id NOT IN (SELECT battery_id FROM read_parquet('{batt_path}'))
    """).fetchone()[0]

    print(f"swap_events -> riders    : {r1:,} unmatched rider_ids")
    print(f"swap_events -> stations  : {r2:,} unmatched station_ids")
    print(f"swap_events -> batteries : {r3:,} unmatched battery_in_ids (non-null)")

    if r1 == 0 and r2 == 0 and r3 == 0:
        print("\n✓ Referential integrity confirmed on cleaned data.")
    else:
        print("\n⚠ Referential integrity issues remain — see FK validation table.")
else:
    print("⚠ One or more Parquet files missing — skipping RI check.")

con.close()

In [ ]:
# 11.5 Preservation of 24 chronology exceptions
print("EXCEPTION PRESERVATION CHECK")
print("=" * 70)

exception_path = INTERIM_DIR / "chronology_exceptions_24_events.parquet"
if exception_path.is_file():
    con = duckdb.connect()
    exc_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{exception_path}')").fetchone()[0]
    exc_in_swaps = con.execute(f"""
        SELECT COUNT(*) FROM read_parquet('{exception_path}') e
        WHERE e.event_id IN (
            SELECT event_id FROM read_parquet('{str(PROCESSED_DIR / 'swap_events_clean.parquet')}')
        )
    """).fetchone()[0]
    con.close()
    print(f"Exceptions file rows : {exc_count:,}")
    print(f"All present in swap_events_clean: {exc_in_swaps} of {exc_count}")
    if exc_in_swaps == exc_count:
        print("✓ All 24 exception events preserved in cleaned swap_events.")
    else:
        print(f"⚠ {exc_count - exc_in_swaps} exception events are missing from cleaned swap_events!")
else:
    print("⚠ Exception file not found — chronology investigation may not have run.")

In [ ]:
# 11.6 DuckDB analytical foundation — verify queries work on processed data
print("ANALYTICAL FOUNDATION SMOKE TEST")
print("=" * 70)

con = duckdb.connect()
swaps_path   = str(PROCESSED_DIR / "swap_events_clean.parquet")

if Path(swaps_path).is_file():
    # 1. Event type distribution
    evt_types = con.execute(f"""
        SELECT event_type, COUNT(*) as events
        FROM read_parquet('{swaps_path}')
        GROUP BY event_type ORDER BY events DESC
    """).fetchdf()
    print("\n1. Event type distribution:")
    display(evt_types)

    # 2. Monthly event volume
    monthly = con.execute(f"""
        SELECT event_month, COUNT(*) as events,
               ROUND(AVG(amount_charged_inr), 2) as avg_revenue
        FROM read_parquet('{swaps_path}')
        WHERE event_month IS NOT NULL
        GROUP BY event_month ORDER BY event_month
        LIMIT 20
    """).fetchdf()
    print("\n2. Monthly swap volume (first 20 months):")
    display(monthly)

    # 3. Confirm dimensions can be joined cleanly
    joined = con.execute(f"""
        SELECT s.city, COUNT(*) as events
        FROM read_parquet('{swaps_path}') se
        JOIN read_parquet('{str(INTERIM_DIR / 'stations_clean.parquet')}') s
          ON se.station_id = s.station_id
        GROUP BY s.city ORDER BY events DESC
    """).fetchdf()
    print("\n3. Events by city (via station join):")
    display(joined)

con.close()

---
## Section 12 · Phase 1 Completion Summary

In [ ]:
# 12.1 Final summary of all files created
print("PHASE 1 COMPLETION SUMMARY")
print("=" * 70)

print("\n[ DATASETS DISCOVERED ]")
if not inventory_df.empty:
    for _, row in inventory_df.iterrows():
        print(f"  ✓ {row['dataset']:<40} {int(row['rows']):>12,} rows")

print("\n[ DATA QUALITY FINDINGS ]")
if not quality_df.empty:
    q = quality_df.set_index("issue")["affected_records"]
    findings = [
        ("Swap events (total)",           q.get("swap_event_rows", 0)),
        ("Swap completed events",          q.get("swap_completed_rows", 0)),
        ("Invalid swap timestamps",        q.get("swap_timestamp_invalid", 0)),
        ("Offline_batch sync events",      q.get("offline_batch_rows", 0)),
        ("Near-duplicate pairs (±120s)",   q.get("offline_near_duplicate_pairs_120s", 0)),
        ("Test station events",            q.get("test_station_events", 0)),
        ("SOH_in > 100%",                  q.get("soh_in_pct_over_100", 0)),
        ("Firmware v3.2.0 candidates",     q.get("firmware_candidate_count", 0)),
        ("Firmware +5h30 hour matched",    q.get("firmware_plus_5h_matched", 0)),
        ("Telemetry invalid hour_start",   q.get("telemetry_timestamp_invalid", 0)),
        ("Ticket CSAT missing",            q.get("ticket_csat_missing", 0)),
    ]
    for label, val in findings:
        print(f"  {label:<45} {int(val):>12,}")

print("\n[ CLEANING PERFORMED ]")
print(f"  Transformations applied: {len(cleaning_log)}")
print("  No raw data modified. All transformations in interim/processed/.")
print("  SOH/SOC anomalies documented but NOT removed (await BMS validation).")
print("  Near-duplicate offline events retained (await business-rule confirmation).")

print("\n[ CHRONOLOGY EXCEPTION — 24 EVENTS ]")
if not df_v320.empty and not df_pre_commission.empty:
    print(f"  Events: {len(df_pre_commission):,}")
    print(f"  Firmware: v3.2.0 exclusively")
    print(f"  Station-days: {df_pre_commission.groupby(['station_id','event_date']).ngroups}")
    print(f"  Finding: event_date = commissioned_date - 1 day; no telemetry on event_date;")
    print(f"           telemetry exists on commissioned_date; event_ts+5h30 matches telemetry hour")
    print(f"  Disposition: PRESERVED as traceable exceptions in interim/chronology_exceptions_24_events.parquet")
else:
    print("  ⚠ Investigation did not complete — check Section 9 outputs.")

print("\n[ FILES CREATED ]")
output_files = [
    # Processed
    ("data/processed/swap_events_clean.parquet",             "Cleaned swap events (Parquet, Snappy)"),
    ("data/processed/station_hourly_status_clean.parquet",   "Cleaned telemetry (Parquet, Snappy)"),
    # Interim
    ("data/interim/fleet_partners_clean.parquet",            "Cleaned fleet partners reference"),
    ("data/interim/stations_clean.parquet",                  "Cleaned stations reference"),
    ("data/interim/city_daily_context_clean.parquet",        "Cleaned daily context reference"),
    ("data/interim/batteries_clean.parquet",                 "Cleaned batteries reference"),
    ("data/interim/riders_clean.parquet",                    "Cleaned riders reference"),
    ("data/interim/support_tickets_clean.parquet",           "Cleaned support tickets"),
    ("data/interim/chronology_exceptions_24_events.parquet", "24 traceable chronology exception events"),
    # Outputs
    ("outputs/tables/dataset_inventory.csv",                 "Dataset inventory (from audit)"),
    ("outputs/tables/missing_value_summary.csv",             "Missing value counts per column"),
    ("outputs/tables/duplicate_key_summary.csv",             "Key uniqueness per dataset"),
    ("outputs/tables/categorical_value_summary.csv",         "Categorical distributions"),
    ("outputs/tables/numerical_summary.csv",                 "Numerical distributions"),
    ("outputs/tables/foreign_key_validation.csv",            "FK validation results"),
    ("outputs/tables/data_quality_issue_summary.csv",        "Quality issue counts"),
    ("outputs/tables/cleaning_log.csv",                      "Transformation log"),
    ("reports/stage_01_data_audit.md",                       "Audit report (from audit_data.py)"),
]

for rel_path, desc in output_files:
    full_path = ROOT_DIR / rel_path
    exists = "✓" if full_path.is_file() else "⚠ MISSING"
    print(f"  {exists}  {rel_path:<60} {desc}")

print("\n[ ANALYTICAL FOUNDATION ]")
print("  Six analytical dimensions supported:")
print("  1. Network & service performance    -> swap_events + station_hourly_status")
print("  2. Customer experience & support    -> support_tickets + riders")
print("  3. Station & city-level operations  -> stations + city_daily_context")
print("  4. Battery performance & degradation -> batteries + swap_events")
print("  5. Pricing & fleet partner economics -> fleet_partners + swap_events + riders")
print("  6. Rider retention & behavior        -> riders + swap_events (time series)")

print("\n[ REMAINING LIMITATIONS ]")
print("  1. SOH/SOC values >100% need BMS instrument definition before cleaning")
print("  2. Offline near-duplicate candidates need business-rule adjudication")
print("  3. Firmware timezone issue (UTC vs IST) is evidence-based but not confirmed")
print("  4. City spelling variants need canonical mapping (if any exist)")
print("  5. Test station events need explicit scope decision before analysis")
print("  6. CSAT non-response may be non-random — treat with caution in Phase 2")

print("\n[ PHASE 2 READINESS ]")
print("  ✓ All 8 datasets present and profiled")
print("  ✓ Relationships validated; cardinality documented")
print("  ✓ 24-event chronology exception reproduced and preserved")
print("  ✓ Cleaned Parquet files available for DuckDB analytical queries")
print("  ✓ Cleaning log ensures reproducibility")
print("  READY FOR PHASE 2 — pending user review and approval.")